# SEC EDGAR Fetch

This notebook fetches recent SEC filings for the selected companies, normalizes the text, and writes a compact corpus to disk.
The user selects a subset of tickers, clicks fetch, and the notebook saves the retrieved filings for downstream chunking and retrieval experiments.

In [1]:
import json
import re
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

STOCKS = [
    {'company': 'Visa Inc.', 'ticker': 'V', 'cik': '1403161'},
    {'company': 'Mastercard Incorporated', 'ticker': 'MA', 'cik': '1141391'}
]
STOCK_LOOKUP = {item['ticker']: item for item in STOCKS}
USER_AGENT = 'sanngesh sanngesh17@gmail.com'
TARGET_FILING_YEAR = 2025


def fetch_sec_company_submissions(cik: str, user_agent: str, timeout: int = 30) -> dict:
    cik_padded = str(cik).strip().zfill(10)
    url = f'https://data.sec.gov/submissions/CIK{cik_padded}.json'
    headers = {
        'User-Agent': user_agent,
        'Accept-Encoding': 'gzip, deflate',
        'Host': 'data.sec.gov'
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    return response.json()


def normalize_text_from_html(html: str) -> str:
    doc = (html or '').lstrip()
    looks_like_xml = doc.startswith('<?xml') or '<xbrl' in doc[:4000].lower()
    parser = 'xml' if looks_like_xml else 'lxml'

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=XMLParsedAsHTMLWarning)
        soup = BeautifulSoup(html, features=parser)

    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text = soup.get_text(separator=' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def select_recent_filings(submissions: dict, forms=('10-K', '10-Q', '8-K'), per_form_limits=None, target_year: int = TARGET_FILING_YEAR) -> pd.DataFrame:
    if per_form_limits is None:
        per_form_limits = {'10-K': 2, '10-Q': 4, '8-K': 8}
    recent = pd.DataFrame(submissions['filings']['recent'])
    recent['filingDate'] = pd.to_datetime(recent['filingDate'], errors='coerce')
    recent['filingDate'] = recent['filingDate'].astype('datetime64[ns]')
    recent = recent[recent['filingDate'].notna()]
    recent = recent[recent['filingDate'].dt.year == int(target_year)]
    recent = recent[recent['form'].isin(forms)].sort_values('filingDate', ascending=False)

    chunks = []
    for form in forms:
        limit = per_form_limits.get(form, 0)
        subset = recent[recent['form'] == form].head(limit).copy()
        if subset.empty:
            continue
        chunks.append(subset)

    if not chunks:
        return recent.head(0).copy()
    return pd.concat(chunks, ignore_index=True)


def filing_archive_url(cik: str, accession_number: str, primary_document: str) -> str:
    cik_no_leading_zeros = str(int(str(cik)))
    accession_clean = accession_number.replace('-', '')
    return f'https://www.sec.gov/Archives/edgar/data/{cik_no_leading_zeros}/{accession_clean}/{primary_document}'


def fetch_filing_text(cik: str, accession_number: str, primary_document: str, user_agent: str, timeout: int = 30) -> str:
    url = filing_archive_url(cik, accession_number, primary_document)
    headers = {'User-Agent': user_agent, 'Accept-Encoding': 'gzip, deflate', 'Host': 'www.sec.gov'}
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    return normalize_text_from_html(response.text)


def build_company_corpus(selected_tickers, user_agent: str = USER_AGENT, target_year: int = TARGET_FILING_YEAR) -> list[dict]:
    records = []
    for ticker in selected_tickers:
        company = STOCK_LOOKUP[ticker]
        submissions = fetch_sec_company_submissions(company['cik'], user_agent=user_agent)
        filings = select_recent_filings(submissions, target_year=target_year)
        submissions_path = DATA_RAW / f'{ticker.lower()}_submissions.json'
        submissions_path.write_text(json.dumps(submissions, indent=2), encoding='utf-8')

        for row in filings.itertuples(index=False):
            try:
                accession_number = str(row.accessionNumber)
                primary_document = str(row.primaryDocument)
                text = fetch_filing_text(company['cik'], accession_number, primary_document, user_agent=user_agent)
            except requests.HTTPError:
                continue
            filing_date_str = str(row.filingDate).split(' ')[0]
            records.append({
                'company': company['company'],
                'ticker': ticker,
                'cik': company['cik'],
                'form': row.form,
                'filing_date': filing_date_str,
                'accession_number': accession_number,
                'primary_document': primary_document,
                'url': filing_archive_url(company['cik'], accession_number, primary_document),
                'text': text
            })
            time.sleep(0.2)
    return records

In [2]:
selected_tickers = ['V', 'MA']
corpus = build_company_corpus(selected_tickers, user_agent=USER_AGENT, target_year=TARGET_FILING_YEAR)
processed_path = DATA_PROCESSED / 'sec_corpus.jsonl'
with processed_path.open('w', encoding='utf-8') as handle:
    for record in corpus:
        handle.write(json.dumps(record) + '\n')

print(f'Saved {len(corpus)} filings for {TARGET_FILING_YEAR} to {processed_path}')

Saved 24 filings for 2025 to j:\Gen AI Assignment - Final\data\processed\sec_corpus.jsonl


The saved JSONL corpus is the input to the next notebook. Each record contains the company metadata, filing metadata, source URL, and normalized full text.